# Preprocessing testing

In [3]:
import dask.dataframe as dd
import os
import pandas as pd
import xarray as xr
from jetstream_interpolate_convcnp.utils.constants import ALTITUDE, GEOPOTENTIAL_HEIGHT, LONGITUDE, LATITUDE, PRESSURE_LEVEL, TIME, WIND_U, WIND_V
import re
import numpy as np

In [3]:
# ecmwf to offgrid parquet

ds_in_path = "/mnt/hdd/jetstream/data/development/ecmwf/july_2019/ecmwf_forecast.nc"
ds = xr.open_dataset(ds_in_path)

level_var_pattern = re.compile(r"^(UGRD|VGRD|HGT)_(\d+)mb$")
target_name_by_source = {
    "UGRD": WIND_U,
    "VGRD": WIND_V,
    "HGT": ALTITUDE,
}

grouped_vars = {WIND_U: [], WIND_V: [], ALTITUDE: []}

for var_name in list(ds.data_vars):
    match = level_var_pattern.match(var_name)
    if not match:
        continue

    source_name, level = match.groups()
    grouped_vars[target_name_by_source[source_name]].append((int(level), var_name))

vars_to_drop = []
for target_name, level_vars in grouped_vars.items():
    if not level_vars:
        continue

    level_vars = sorted(level_vars, key=lambda item: item[0])
    arrays = [
        ds[var_name].expand_dims({PRESSURE_LEVEL: [level]})
        for level, var_name in level_vars
    ]
    ds[target_name] = xr.concat(arrays, dim=PRESSURE_LEVEL)
    vars_to_drop.extend([var_name for _, var_name in level_vars])

if vars_to_drop:
    ds = ds.drop_vars(vars_to_drop)

rename_map = {}
if "latitude" in ds.coords and LATITUDE not in ds.coords:
    rename_map["latitude"] = LATITUDE
if "longitude" in ds.coords and LONGITUDE not in ds.coords:
    rename_map["longitude"] = LONGITUDE
if "time" in ds.coords and TIME != "time" and TIME not in ds.coords:
    rename_map["time"] = TIME

if rename_map:
    ds = ds.rename(rename_map)

In [13]:
# load dask dataframe from parquet
path = "/mnt/hdd/jetstream/data/processed/train_v0.1/datasets/ecmwf/altitude_band=*/coarse_lat=38/coarse_lon=108/*.parquet"
df = dd.read_parquet(path)

df.head()

,index,lat,lon,time,pressure_level,u,v,altitude,log_altitude,u_mean,u_std,v_mean,v_std,u_normed,v_normed,altitude_band,coarse_lat,coarse_lon
0,115335004,38.0,108.0,2019-07-01 00:00:00,700,4.936361,-1.261200,3116.705566,8.044532,1.639245,3.989331,0.006581,4.982009,0.826484,-0.254472,1,38,108
1,115335009,38.0,108.0,2019-07-01 06:00:00,700,5.383106,1.863091,3119.803223,8.045526,1.639245,3.989331,0.006581,4.982009,0.938469,0.372643,1,38,108
2,115335014,38.0,108.0,2019-07-01 12:00:00,700,1.727571,1.038815,3115.796631,8.044240,1.639245,3.989331,0.006581,4.982009,0.022141,0.207192,1,38,108
3,115335019,38.0,108.0,2019-07-01 18:00:00,700,0.689180,1.247330,3125.715820,8.047419,1.639245,3.989331,0.006581,4.982009,-0.238151,0.249046,1,38,108
4,115335024,38.0,108.0,2019-07-02 00:00:00,700,0.234375,3.104984,3132.353271,8.049540,1.639245,3.989331,0.006581,4.982009,-0.352157,0.621918,1,38,108


In [17]:
df[
    (df['time'] == '2019-07-01 06:00:00') &
    (df['lat'] == 38.0) &
    (df['lon'] == 108.0)
].compute()

,index,lat,lon,time,pressure_level,u,v,altitude,log_altitude,u_mean,u_std,v_mean,v_std,u_normed,v_normed,altitude_band,coarse_lat,coarse_lon
1,115335009,38.0,108.0,2019-07-01 06:00:00,700,5.383106,1.863091,3119.803223,8.045526,1.639245,3.989331,0.006581,4.982009,0.938469,0.372643,1,38,108
1,115335008,38.0,108.0,2019-07-01 06:00:00,500,6.644924,-3.394001,5815.170898,8.668225,6.314979,6.113304,-1.406022,5.977722,0.053972,-0.332565,2,38,108
1,115335007,38.0,108.0,2019-07-01 06:00:00,300,23.336536,-8.383713,9567.152344,9.166091,17.275188,9.189481,-0.545651,10.809060,0.659596,-0.725138,4,38,108
1,115335006,38.0,108.0,2019-07-01 06:00:00,250,43.071449,-14.821716,10815.552734,9.288740,22.460773,10.373758,-0.920252,12.549461,1.986809,-1.107734,5,38,108
1,115335005,38.0,108.0,2019-07-01 06:00:00,200,52.916325,-22.536812,12301.153320,9.417448,26.925402,10.511456,-2.027426,12.836745,2.472628,-1.597709,6,38,108


In [92]:
# load the ecmwf dataset as a test

dataset_path = "/mnt/hdd/jetstream/data/processed/train_v0.1/datasets/amdar/*/*/*/*/*/*.parquet"

dataset_df = dd.read_parquet(dataset_path)

In [94]:
dataset_df[
    (dataset_df['time'] == '2019-07-01 01:58:00') &
    (abs(dataset_df['lat'] - 38.183) < 1) &
    (abs(dataset_df['lon'] - 108.417) < 1) &
    (dataset_df['coarse_lat'] == 38.0) &
    (dataset_df['coarse_lon'] == 108.0)
].compute()

,time,date,lat,lon,altitude,u,v,altitude_band,log_altitude,u_normed,v_normed,year,month,day,coarse_lat,coarse_lon
5,2019-07-01 01:58:00,2019-07-01,38.133,108.417,7500.0,16.108835,-6.837795,3,8.922658,<NA>,<NA>,2019,7,1,38,108
6,2019-07-01 01:58:00,2019-07-01,38.133,108.5,7560.0,16.804448,-6.450623,3,8.930626,<NA>,<NA>,2019,7,1,38,108
7,2019-07-01 01:58:00,2019-07-01,38.133,108.533,7620.0,17.964853,-6.185795,3,8.938532,<NA>,<NA>,2019,7,1,38,108


In [25]:
amdar_raw_path = "/mnt/hdd/jetstream/data/development/ceda/july_2019/data.ceda.ac.uk/badc/ukmo-metdb/data/amdars_debug/2019/07/ukmo-metdb_amdars_20190701.csv"

amdar_raw_df = pd.read_csv(amdar_raw_path, skiprows=0, encoding_errors='ignore')

In [83]:
df_debug = amdar_raw_df[
    (amdar_raw_df[amdar_raw_df.columns[0]] == 2019) &
    (amdar_raw_df[amdar_raw_df.columns[1]] == 7.0) &
    (amdar_raw_df[amdar_raw_df.columns[2]] == 1.0) &
    (amdar_raw_df[amdar_raw_df.columns[3]] == 1.0) &
    (amdar_raw_df[amdar_raw_df.columns[4]] == 58.0) &
    (abs(amdar_raw_df[amdar_raw_df.columns[13]] - 38.183) < 1) &
    (abs(amdar_raw_df[amdar_raw_df.columns[14]] - 108.417) < 1)
][amdar_raw_df.columns[0:5].append(amdar_raw_df.columns[13:15]).append(amdar_raw_df.columns[24:26])]

df_debug['u'] = -df_debug[amdar_raw_df.columns[25]] * np.sin(np.radians(df_debug[amdar_raw_df.columns[24]]))
df_debug['v'] = -df_debug[amdar_raw_df.columns[25]] * np.cos(np.radians(df_debug[amdar_raw_df.columns[24]]))

In [84]:
df_debug

,1,2,3,4,5,14,15,25,26,u,v
78186,2019,7.0,1.0,1.0,58.0,38.133,108.417,293.0,17.5,16.108835,-6.837795
78187,2019,7.0,1.0,1.0,58.0,38.133,108.500,291.0,18.0,16.804448,-6.450623
78188,2019,7.0,1.0,1.0,58.0,38.133,108.533,289.0,19.0,17.964853,-6.185795


In [90]:
source_to_target = {
            '1': 'year',
            '2': 'month',
            '3': 'day',
            '4': 'hour',
            '5': 'minute',
            '6': 'second',
            '14': LATITUDE,
            '15': LONGITUDE,
            '16': ALTITUDE,
            '25': 'wind_direction',
            '26': 'wind_speed',
        }

amdar_processed_df = amdar_raw_df.rename(columns=source_to_target)

In [91]:
amdar_processed_df[['year', 'month', 'day', 'hour', 'minute', 'second', LATITUDE, LONGITUDE, ALTITUDE, 'wind_speed', 'wind_direction']]

,year,month,day,hour,minute,second,lat,lon,altitude,wind_speed,wind_direction
0,2019,7.0,1.0,0.0,0.0,-9999999.0,38.183,108.417,8410.0,21.6,293.0
1,2019,7.0,1.0,0.0,0.0,-9999999.0,39.455,-111.985,12190.0,29.8,205.0
2,2019,7.0,1.0,0.0,0.0,-9999999.0,31.700,118.867,7500.0,17.5,259.0
3,2019,7.0,1.0,0.0,0.0,-9999999.0,37.078,-121.507,5800.0,12.9,241.0
4,2019,7.0,1.0,0.0,0.0,-9999999.0,29.013,-82.072,3980.0,8.7,293.0
...,...,...,...,...,...,...,...,...,...,...,...
99995,2019,7.0,1.0,2.0,33.0,21.0,-0.117,-78.350,2290.0,3.1,136.0
99996,2019,7.0,1.0,2.0,33.0,21.0,6.150,-75.417,2160.0,4.6,107.0
99997,2019,7.0,1.0,2.0,33.0,23.0,35.072,-106.247,12160.0,15.9,342.0
99998,2019,7.0,1.0,2.0,33.0,23.0,22.977,-154.693,10360.0,14.9,286.0
